In [1]:
import geneformer
import inspect

print("=" * 70)
print("GENEFORMER VERSION / TOKENIZER CHECK")
print("=" * 70)

print("Geneformer location:")
print(geneformer.__file__)

print("\nGeneformer version:")
print(getattr(geneformer, "__version__", "version not exposed"))

from geneformer import TranscriptomeTokenizer

print("\nTranscriptomeTokenizer:")
print(TranscriptomeTokenizer)

print("\nConstructor:")
print(inspect.signature(TranscriptomeTokenizer))

F:\GeneformerProject\geneformer_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1011 18:48:27.752000 4808 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


GENEFORMER VERSION / TOKENIZER CHECK
Geneformer location:
F:\GeneformerProject\Geneformer\geneformer\__init__.py

Geneformer version:
version not exposed

TranscriptomeTokenizer:
<class 'geneformer.tokenizer.TranscriptomeTokenizer'>

Constructor:
(custom_attr_name_dict=None, nproc=1, chunk_size=512, model_input_size=4096, special_token=True, collapse_gene_ids=True, use_h5ad_index=False, keep_counts=False, model_version='V2', gene_median_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/gene_median_dictionary_gc104M.pkl'), token_dictionary_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/token_dictionary_gc104M.pkl'), gene_mapping_file=WindowsPath('F:/GeneformerProject/Geneformer/geneformer/ensembl_mapping_dict_gc104M.pkl'))


In [2]:
# ============================================================
# STEP B — SHAP-BASED GENE SELECTION + GENEFORMER TOKENIZATION
#
# Pipeline:
# Training data
#       ↓
# Logistic Regression
#       ↓
# SHAP gene importance
#       ↓
# Top 512 / 1024 / 2048 / 4096 genes
#       ↓
# Geneformer expression ranking
#       ↓
# Geneformer token sequences
#       ↓
# Hugging Face Dataset
# ============================================================

import pandas as pd
import numpy as np
import pickle
import json
import joblib

from pathlib import Path
from tqdm.auto import tqdm

from datasets import Dataset, DatasetDict

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

import shap


In [5]:
# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = Path(
    r"F:\GeneformerProject\data\geneformer_ready\preprocessed_5_cancers"
)

OUTPUT_DIR = Path(
    r"F:\GeneformerProject\data\geneformer_tokenized\second_strategy\preprocessed_5_cancers"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TOKEN_FILE = Path(
    r"F:\GeneformerProject\Geneformer\geneformer"
    r"\token_dictionary_gc104M.pkl"
)


# ============================================================
# 2. PARAMETERS
# ============================================================

TOP_K = [512, 1024, 2048, 4096]

MAX_LENGTH = TOP_K

# 4096 = CLS + 4094 genes + EOS
N_GENES = [510, 1022, 2046, 4094]

LABEL_COLUMNS = [
    "Cancer_Type",
    "tissue_type"
]

TRAIN_FILE = BASE_DIR / "train_geneformer.parquet"
VAL_FILE   = BASE_DIR / "validation_geneformer.parquet"
TEST_FILE  = BASE_DIR / "test_geneformer.parquet"


In [6]:
# ============================================================
# 3. LOAD GENEFORMER VOCABULARY
# ============================================================

with open(TOKEN_FILE, "rb") as f:
    token_dictionary = pickle.load(f)

CLS_TOKEN = token_dictionary["<cls>"]
EOS_TOKEN = token_dictionary["<eos>"]

print("=" * 70)
print("SHAP-BASED GENE SELECTION + GENEFORMER TOKENIZATION")
print("=" * 70)

print("Geneformer vocabulary size:", len(token_dictionary))
print("CLS token:", CLS_TOKEN)
print("EOS token:", EOS_TOKEN)
print("Maximum sequence length:", MAX_LENGTH)
print("Genes per sequence:", N_GENES)


# ============================================================
# 4. LOAD DATA
# ============================================================

print("\nLoading datasets...")

train_df = pd.read_parquet(TRAIN_FILE)
val_df   = pd.read_parquet(VAL_FILE)
test_df  = pd.read_parquet(TEST_FILE)

print("\nDataset sizes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# ============================================================
# 5. IDENTIFY GENES
# ============================================================

gene_columns = [
    c for c in train_df.columns
    if c not in LABEL_COLUMNS
]

print("\nNumber of genes:", len(gene_columns))

assert len(gene_columns) == 15100, (
    f"Expected 15100 genes, found {len(gene_columns)}"
)


# Make sure validation/test have exactly the same genes

val_gene_columns = [
    c for c in val_df.columns
    if c not in LABEL_COLUMNS
]

test_gene_columns = [
    c for c in test_df.columns
    if c not in LABEL_COLUMNS
]

assert gene_columns == val_gene_columns
assert gene_columns == test_gene_columns

print("✓ Train/validation/test gene columns match")


# ============================================================
# 6. CHECK GENEFORMER VOCABULARY
# ============================================================

missing_genes = [
    gene for gene in gene_columns
    if gene not in token_dictionary
]

print(
    "Genes missing from Geneformer vocabulary:",
    len(missing_genes)
)

assert len(missing_genes) == 0, (
    f"{len(missing_genes)} genes are missing "
    "from Geneformer vocabulary."
)

print("✓ All genes are present in Geneformer vocabulary")


# ============================================================
# 7. CREATE CANCER LABEL MAPPING
# ============================================================

# IMPORTANT:
# Class mapping is created ONLY from training data.

cancer_classes = sorted(
    train_df["Cancer_Type"]
    .dropna()
    .unique()
    .tolist()
)

label_to_id = {
    label: idx
    for idx, label in enumerate(cancer_classes)
}

id_to_label = {
    idx: label
    for label, idx in label_to_id.items()
}

print("\nNumber of cancer classes:", len(cancer_classes))

print("\nCancer class mapping:")

for idx, label in id_to_label.items():
    print(f"{idx:3d} -> {label}")


# ============================================================
# 8. CHECK VALIDATION / TEST LABELS
# ============================================================

unknown_val_labels = sorted(
    set(val_df["Cancer_Type"]) - set(label_to_id)
)

unknown_test_labels = sorted(
    set(test_df["Cancer_Type"]) - set(label_to_id)
)

assert len(unknown_val_labels) == 0, (
    "Validation contains cancer classes "
    "not present in training."
)

assert len(unknown_test_labels) == 0, (
    "Test contains cancer classes "
    "not present in training."
)


# ============================================================
# 9. PREPARE TRAINING DATA FOR SHAP
# ============================================================

print("\n" + "=" * 70)
print("STEP 1 — TRAIN LOGISTIC REGRESSION FOR SHAP")
print("=" * 70)

X_train = train_df[gene_columns].to_numpy(
    dtype=np.float32
)

y_train = train_df["Cancer_Type"].map(
    label_to_id
).to_numpy(
    dtype=np.int32
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


# Check values

assert np.all(np.isfinite(X_train)), (
    "Training data contains NaN or infinite values."
)


SHAP-BASED GENE SELECTION + GENEFORMER TOKENIZATION
Geneformer vocabulary size: 20275
CLS token: 2
EOS token: 3
Maximum sequence length: [512, 1024, 2048, 4096]
Genes per sequence: [510, 1022, 2046, 4094]

Loading datasets...

Dataset sizes:
Train: (2411, 15102)
Validation: (517, 15102)
Test: (517, 15102)

Number of genes: 15100
✓ Train/validation/test gene columns match
Genes missing from Geneformer vocabulary: 0
✓ All genes are present in Geneformer vocabulary

Number of cancer classes: 5

Cancer class mapping:
  0 -> BRCA
  1 -> COAD
  2 -> KIRC
  3 -> LUAD
  4 -> PRAD

STEP 1 — TRAIN LOGISTIC REGRESSION FOR SHAP
X_train shape: (2411, 15100)
y_train shape: (2411,)


In [7]:
# ============================================================
# 10. TRAIN LOGISTIC REGRESSION
# ============================================================

print("\nTraining Logistic Regression...")

shap_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs"
)

shap_model.fit(
    X_train,
    y_train
)

print("✓ Logistic Regression trained")
print("Number of classes:", len(shap_model.classes_))
print("Coefficient shape:", shap_model.coef_.shape)



Training Logistic Regression...
✓ Logistic Regression trained
Number of classes: 5
Coefficient shape: (5, 15100)


In [8]:
# ============================================================
# 11. CREATE BALANCED SHAP SUBSET FROM TRAINING DATA
# ============================================================

print("\n" + "=" * 70)
print("STEP 2 — CREATE BALANCED SHAP SUBSET")
print("=" * 70)


SHAP_SAMPLES_PER_CLASS = 100


shap_subset_list = []


for cancer_type, group in train_df.groupby("Cancer_Type"):

    n_select = min(
        SHAP_SAMPLES_PER_CLASS,
        len(group)
    )

    sampled = group.sample(
        n=n_select,
        random_state=42
    )

    shap_subset_list.append(
        sampled
    )


shap_df = pd.concat(
    shap_subset_list,
    axis=0
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print(
    "SHAP subset shape:",
    shap_df.shape
)


print(
    "\nClass distribution:"
)

print(
    shap_df["Cancer_Type"].value_counts()
)


# ============================================================
# 12. PREPARE SHAP INPUT
# ============================================================


X_shap = shap_df[
    gene_columns
].to_numpy(
    dtype=np.float32
)


print(
    "\nSHAP input:",
    X_shap.shape
)


# ============================================================
# 13. SHAP EXPLAINER
# ============================================================

print("\n" + "=" * 70)
print("STEP 3 — CALCULATE SHAP VALUES")
print("=" * 70)


explainer = shap.LinearExplainer(
    shap_model,
    X_train
)


print(
    "Calculating SHAP values on balanced subset..."
)


shap_values = explainer.shap_values(
    X_shap
)


print(
    "SHAP calculation complete."
)


print(
    "SHAP object type:",
    type(shap_values)
)


# ============================================================
# 14. CONVERT SHAP VALUES TO ARRAY
# ============================================================


if isinstance(shap_values, list):

    print(
        "SHAP returned list:",
        len(shap_values),
        "classes"
    )


    shap_array = np.stack(
        shap_values,
        axis=-1
    )


else:

    shap_array = np.asarray(
        shap_values
    )


    if shap_array.ndim == 3:

        if shap_array.shape[0] == len(shap_model.classes_):

            shap_array = np.transpose(
                shap_array,
                (1,2,0)
            )


print(
    "Final SHAP array shape:",
    shap_array.shape
)
# ============================================================
# 15. GLOBAL SHAP IMPORTANCE
# ============================================================

print("\n" + "=" * 70)
print("STEP 3 — GLOBAL GENE RANKING")
print("=" * 70)

# Mean absolute SHAP across:
#
#   samples
#   cancer classes
#
# Result:
#
#   one importance value per gene

mean_abs_shap = np.mean(
    np.abs(shap_array),
    axis=(0, 2)
)

assert mean_abs_shap.shape[0] == len(gene_columns)

print(
    "Number of SHAP-ranked genes:",
    len(mean_abs_shap)
)


# ============================================================
# 16. CREATE SHAP RANKING TABLE
# ============================================================

shap_ranking = pd.DataFrame({
    "gene": gene_columns,
    "mean_abs_shap": mean_abs_shap
})

shap_ranking = shap_ranking.sort_values(
    "mean_abs_shap",
    ascending=False
).reset_index(drop=True)

shap_ranking["rank"] = (
    np.arange(len(shap_ranking)) + 1
)

print("\nTop 20 SHAP-ranked genes:")

print(
    shap_ranking[
        ["rank", "gene", "mean_abs_shap"]
    ].head(20).to_string(index=False)
)


# ============================================================
# 17. SAVE COMPLETE SHAP RANKING
# ============================================================

SHAP_RANKING_FILE = (
    OUTPUT_DIR / "shap_gene_ranking.csv"
)

shap_ranking.to_csv(
    SHAP_RANKING_FILE,
    index=False
)

print(
    "\nSaved SHAP ranking:",
    SHAP_RANKING_FILE
)


# ============================================================
# 18. SAVE SHAP MODEL
# ============================================================

SHAP_MODEL_FILE = (
    OUTPUT_DIR / "shap_logistic_regression.joblib"
)

joblib.dump(
    shap_model,
    SHAP_MODEL_FILE
)

print(
    "Saved SHAP Logistic Regression:",
    SHAP_MODEL_FILE
)


# ============================================================
# 19. SAVE SHAP INFORMATION
# ============================================================

SHAP_INFO_FILE = (
    OUTPUT_DIR / "shap_info.json"
)

shap_info = {
    "method": "Logistic Regression + SHAP",
    "number_of_training_samples": int(len(train_df)),
    "number_of_genes": int(len(gene_columns)),
    "number_of_classes": int(len(cancer_classes)),
    "top_k": TOP_K,
    "geneformer_max_length": MAX_LENGTH,
    "geneformer_genes_per_sequence": N_GENES
}

with open(
    SHAP_INFO_FILE,
    "w"
) as f:

    json.dump(
        shap_info,
        f,
        indent=4
    )
# ============================================================
# 20. PRECOMPUTE TOKEN IDS
# ============================================================

gene_token_ids = {
    gene: token_dictionary[gene]
    for gene in gene_columns
}

print(
    "\nPrecomputed token IDs for",
    len(gene_token_ids),
    "genes."
)


# ============================================================
# 21. CREATE SHAP GENE LISTS
# ============================================================

selected_gene_lists = {}

for k in N_GENES:

    selected_genes = (
        shap_ranking
        .head(k)["gene"]
        .tolist()
    )

    selected_gene_lists[k] = selected_genes

    print(
        f"\nTop {k} SHAP genes:"
    )

    print(
        selected_genes[:10],
        "..."
    )

    # Save gene list

    gene_file = (
        OUTPUT_DIR /
        f"top_{k}_shap_genes.csv"
    )

    pd.DataFrame({
        "rank": np.arange(1, k + 1),
        "gene": selected_genes
    }).to_csv(
        gene_file,
        index=False
    )



STEP 2 — CREATE BALANCED SHAP SUBSET
SHAP subset shape: (500, 15102)

Class distribution:
Cancer_Type
LUAD    100
BRCA    100
COAD    100
PRAD    100
KIRC    100
Name: count, dtype: int64

SHAP input: (500, 15100)

STEP 3 — CALCULATE SHAP VALUES
Calculating SHAP values on balanced subset...
SHAP calculation complete.
SHAP object type: <class 'numpy.ndarray'>
Final SHAP array shape: (500, 15100, 5)

STEP 3 — GLOBAL GENE RANKING
Number of SHAP-ranked genes: 15100

Top 20 SHAP-ranked genes:
 rank            gene  mean_abs_shap
    1 ENSG00000142515       0.060008
    2 ENSG00000168878       0.058416
    3 ENSG00000105388       0.051955
    4 ENSG00000167751       0.051192
    5 ENSG00000185303       0.049035
    6 ENSG00000086548       0.045412
    7 ENSG00000129824       0.044452
    8 ENSG00000101076       0.041965
    9 ENSG00000006611       0.040701
   10 ENSG00000110484       0.038239
   11 ENSG00000122852       0.037608
   12 ENSG00000159184       0.036434
   13 ENSG00000131400    

In [9]:
# ============================================================
# 20. TOKENIZATION FUNCTION
# ============================================================

def tokenize_dataframe(
    df,
    dataset_name,
    selected_genes,
    target_k
):

    print("\n" + "=" * 70)
    print(
        f"Tokenizing {dataset_name} "
        f"using TOP {target_k} SHAP GENES"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Get only SHAP-selected genes
    # --------------------------------------------------------

    X = df[selected_genes].to_numpy(
        dtype=np.float32
    )

    input_ids = []

    labels = []

    cancer_types = []
    tissue_types = []

    # --------------------------------------------------------
    # Process samples
    # --------------------------------------------------------

    for i in tqdm(
        range(X.shape[0]),
        desc=dataset_name
    ):

        values = X[i]

        # ----------------------------------------------------
        # Check values
        # ----------------------------------------------------

        valid = np.isfinite(values)

        if not np.all(valid):

            raise ValueError(
                f"{dataset_name}, sample {i}: "
                "contains NaN or infinite values."
            )

        # ----------------------------------------------------
        # Rank SHAP-selected genes
        #
        # IMPORTANT:
        # SHAP determines WHICH genes are allowed.
        #
        # Geneformer then ranks those genes by expression
        # for each individual sample.
        # ----------------------------------------------------

        order = np.argsort(
            -values,
            kind="stable"
        )

        ranked_genes = [
            selected_genes[j]
            for j in order
        ]

        # ----------------------------------------------------
        # Convert genes to Geneformer tokens
        # ----------------------------------------------------

        ranked_tokens = np.array(
            [
                gene_token_ids[gene]
                for gene in ranked_genes
            ],
            dtype=np.int32
        )

        # ----------------------------------------------------
        # ----------------------------------------------------

        sequence = np.concatenate(
            [
                np.array(
                    [CLS_TOKEN],
                    dtype=np.int32
                ),

                ranked_tokens,

                np.array(
                    [EOS_TOKEN],
                    dtype=np.int32
                )
            ]
        )

        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        expected_length = target_k + 2

        assert len(sequence) == expected_length

        assert sequence[0] == CLS_TOKEN
        assert sequence[-1] == EOS_TOKEN

        input_ids.append(
            sequence.tolist()
        )

        # ----------------------------------------------------
        # Cancer label
        # ----------------------------------------------------

        cancer_type = df.iloc[i]["Cancer_Type"]

        labels.append(
            label_to_id[cancer_type]
        )

        cancer_types.append(
            cancer_type
        )

        tissue_types.append(
            df.iloc[i]["tissue_type"]
        )

    # --------------------------------------------------------
    # Create Hugging Face Dataset
    # --------------------------------------------------------

    dataset_dict = {
        "input_ids": input_ids,
        "labels": labels,
        "Cancer_Type": cancer_types,
        "tissue_type": tissue_types
    }

    dataset = Dataset.from_dict(
        dataset_dict
    )

    return dataset


In [10]:
# ============================================================
# 21. CREATE FOUR GENEFORMER DATASETS
# ============================================================

for k in N_GENES:

    print("\n\n")
    print("#" * 70)
    print(
        f"PROCESSING SHAP TOP {k}"
    )
    print("#" * 70)

    selected_genes = selected_gene_lists[k]

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    train_dataset = tokenize_dataframe(
        train_df,
        "TRAIN",
        selected_genes,
        k
    )

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    validation_dataset = tokenize_dataframe(
        val_df,
        "VALIDATION",
        selected_genes,
        k
    )

    # --------------------------------------------------------
    # TEST
    # --------------------------------------------------------

    test_dataset = tokenize_dataframe(
        test_df,
        "TEST",
        selected_genes,
        k
    )

    # --------------------------------------------------------
    # DatasetDict
    # --------------------------------------------------------

    tokenized_dataset = DatasetDict({
        "train": train_dataset,
        "validation": validation_dataset,
        "test": test_dataset
    })

    # --------------------------------------------------------
    # Output path
    # --------------------------------------------------------

    FINAL_OUTPUT = (
        OUTPUT_DIR /
        f"tcga_geneformer_shap_top{k}"
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    tokenized_dataset.save_to_disk(
        str(FINAL_OUTPUT)
    )

    print(
        "\nSaved:",
        FINAL_OUTPUT
    )

    # --------------------------------------------------------
    # Verification
    # --------------------------------------------------------

    print("\nDataset:")
    print(tokenized_dataset)

    print("\nSizes:")
    print(
        "Train:",
        len(tokenized_dataset["train"])
    )

    print(
        "Validation:",
        len(tokenized_dataset["validation"])
    )

    print(
        "Test:",
        len(tokenized_dataset["test"])
    )

    sample = tokenized_dataset["train"][0]

    print("\nFirst sample:")
    print(
        "Cancer:",
        sample["Cancer_Type"]
    )

    print(
        "Label:",
        sample["labels"]
    )

    print(
        "Sequence length:",
        len(sample["input_ids"])
    )

    print(
        "First tokens:",
        sample["input_ids"][:10]
    )

    print(
        "Last tokens:",
        sample["input_ids"][-10:]
    )


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SHAP + GENEFORMER PIPELINE COMPLETE")
print("=" * 70)

print("\nSHAP ranking:")
print(
    SHAP_RANKING_FILE
)

print("\nGenerated Geneformer datasets:")

for k in TOP_K:

    print(
        f"  TOP {k}: "
        f"{OUTPUT_DIR / f'tcga_geneformer_shap_top{k}'}"
    )

print("\n✓ SHAP calculated using TRAINING DATA only")
print("✓ Global mean |SHAP| gene ranking created")
print("✓ Top 512 genes selected")
print("✓ Top 1024 genes selected")
print("✓ Top 2048 genes selected")
print("✓ Top 4096 genes selected")
print("✓ Same gene lists applied to validation/test")
print("✓ Geneformer tokenization completed")
print("✓ Hugging Face datasets saved")




######################################################################
PROCESSING SHAP TOP 510
######################################################################

Tokenizing TRAIN using TOP 510 SHAP GENES


TRAIN: 100%|██████████| 2411/2411 [00:23<00:00, 102.39it/s]



Tokenizing VALIDATION using TOP 510 SHAP GENES


VALIDATION: 100%|██████████| 517/517 [00:04<00:00, 109.51it/s]



Tokenizing TEST using TOP 510 SHAP GENES


...


Tokenizing VALIDATION using TOP 4094 SHAP GENES


VALIDATION: 100%|██████████| 517/517 [00:05<00:00, 89.48it/s] 



Tokenizing TEST using TOP 4094 SHAP GENES


TEST: 100%|██████████| 517/517 [00:05<00:00, 97.24it/s] 



Saved: F:\GeneformerProject\data\geneformer_tokenized\second_strategy\preprocessed_5_cancers\tcga_geneformer_shap_top4094

Dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 2411
    })
    validation: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 517
    })
    test: Dataset({
        features: ['input_ids', 'labels', 'Cancer_Type', 'tissue_type'],
        num_rows: 517
    })
})

Sizes:
Train: 2411
Validation: 517
Test: 517

First sample:
Cancer: PRAD
Label: 4
Sequence length: 4096
First tokens: [2, 8128, 11973, 14988, 10099, 10219, 2899, 3558, 11801, 16068]
Last tokens: [12701, 7584, 12075, 12198, 15044, 6606, 3546, 13378, 2415, 3]

SHAP + GENEFORMER PIPELINE COMPLETE

SHAP ranking:
F:\GeneformerProject\data\geneformer_tokenized\second_strategy\preprocessed_5_cancers\shap_gene_ranking.csv

Generated Geneformer datasets:
  TOP 512: F:\GeneformerP